In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/google/gemma-4-E4B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-4-E4B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Load model directly
import torch
import gc

# Clear previous model from memory to avoid OOM
if 'model' in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/gemma-4-E4B")
model = AutoModelForMultimodalLM.from_pretrained(
    "google/gemma-4-E4B",
    device_map="cuda",  # Force it entirely onto the GPU
    dtype=torch.float16
)


In [ ]:
user_input = "Hello, how are you today? what is your favorite color?"

# Manually format the prompt using Gemma's specific chat tokens
prompt = f"<start_of_turn>user\n{user_input}<end_of_turn>\n<start_of_turn>model\n"

# Process the prompt
inputs = processor(text=prompt, return_tensors="pt").to(model.device)

# Generate response
outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

# Decode only the newly generated tokens
input_length = inputs["input_ids"].shape[1]
response_text = processor.decode(outputs[0][input_length:], skip_special_tokens=True)

# Truncate any hallucinated follow-up turns
if "<end_of_turn>" in response_text:
    response_text = response_text.split("<end_of_turn>")[0]

print("Assistant:", response_text.strip())


### Fine-Tuning Setup (LoRA)
First, install the necessary libraries for Parameter-Efficient Fine-Tuning.

In [ ]:
!pip install -q peft bitsandbytes trl accelerate datasets

In [ ]:
from peft import LoraConfig, get_peft_model
import torch

# Define the LoRA configuration
# We target the attention projections which is standard practice
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the existing model with the PEFT adapters
# (Note: For best memory efficiency, the base model should ideally be loaded in 4-bit using BitsAndBytesConfig first)
peft_model = get_peft_model(model, peft_config)

# Print the number of trainable parameters. It should be a very small percentage!
peft_model.print_trainable_parameters()


### Next Steps
From here, you would prepare your custom dataset using the `datasets` library, tokenize it with the `processor`, and pass it to a `Trainer`.

```python
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./gemma_adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, # Adjust based on your dataset size
    fp16=True,
)

# trainer = Trainer(model=peft_model, args=training_args, train_dataset=your_dataset, data_collator=your_collator)
# trainer.train()
```

In [ ]:
import sys, platform, os
print("python:", sys.version.split()[0], platform.platform())
print("cwd:", os.getcwd())
print("is_colab:", 'google.colab' in sys.modules or os.path.exists('/content'))
try:
    import torch
    print("torch:", torch.__version__, "cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info()
        print(f"gpu_mem_free_GB: {free/1e9:.1f} / total_GB: {total/1e9:.1f}")
except Exception as e:
    print("torch: not importable ->", repr(e))
for name in ("model", "processor", "peft_model"):
    print(f"{name}_in_globals:", name in globals())